# Chapter 10: Trees

Trees model parent–child hierarchy or organise keys for search. Build a reusable binary search tree, extend it into an AVL tree, then implement left-leaning red-black insertion. Apply traversal to student records, directories, and a decision stump.

**Prerequisites:** functions, classes, arrays, and recursion.

Select **JavaScript (Node.js)** and run all cells from the top. This notebook adapts the supplied TypeScript guide to executable JavaScript; no compiler or external packages are required. Restart the kernel before rerunning all declarations. All 12 checkpoints and all three main exercises have runnable worked solutions.

TypeScript’s generic types, readonly fields, access modifiers, and union types describe compile-time contracts in the guide. They are not enforced by JavaScript. Here, use consistent comparators, finite numeric keys, valid record shapes, and supported traversal orders. Equal comparator keys keep the existing value. Do not mutate ordering keys or tree links during a traversal.

## 1. Find a value in a three-node tree

A BST has at most two children per node. Every key in the left subtree is smaller; every key in the right subtree is larger. Missing children are null. Search follows one branch until it finds the key or an empty position.

In [1]:
const contains = (node, target)=>{
    while(node !== null){
        if (target === node.value) return true;
        node = target < node.value ? node.left : node.right;
    }
    return false;
};
const simpleRoot = {
    value: 11,
    left: {
        value: 7,
        left: null,
        right: null
    },
    right: {
        value: 15,
        left: null,
        right: null
    }
};
console.log(contains(simpleRoot, 15));
console.log(contains(simpleRoot, 9));

true
false


### Checkpoint 1

What does contains(simpleRoot, 11) return, and how many nodes are compared?

Predict the result before running the worked solution.

**Worked solution.** True after one node comparison, because 11 is the root.

In [ ]:
(() => {
const assert = require("node:assert/strict");
assert.equal(contains(simpleRoot, 11), true);
let visits = 0;
const countedRoot = { get value() { visits++; return 11; }, left: null, right: null };
assert.equal(contains(countedRoot, 11), true);
assert.equal(visits, 1);
console.log('Root found after', visits, 'comparison.');
})();

## 2. Grow the same tree to seven values

| Parent | Left | Right |
| --- | --- | --- |
| 11 | 7 | 15 |
| 7 | 5 | 9 |
| 15 | 13 | 20 |

11 is the root; 5, 9, 13, and 20 are leaves. The ancestors of 9 are 7 and 11. The subtree at 7 contains 7, 5, and 9. Depth counts edges from the root; height counts edges to the farthest leaf. Empty subtrees have height −1, leaves height 0. The full seven-node tree has height 2.

In [ ]:
const originalValues = [11, 7, 15, 5, 9, 13, 20];
function nodeHeight(node) {
 return node === null ? -1 : 1 + Math.max(nodeHeight(node.left), nodeHeight(node.right));
}

### Checkpoint 2

Find the depth of root 11 and the height of leaf 9.

Predict the result before running the worked solution.

**Worked solution.** Both are 0 because neither measurement needs an edge.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const leaf9 = { value: 9, left: null, right: null };
assert.equal(nodeHeight(leaf9), 0);
assert.equal(nodeHeight(null), -1);
const rootDepth = 0;
assert.equal(rootDepth, 0);
console.log({ depthOf11: rootDepth, heightOf9: nodeHeight(leaf9) });
})();

## 3. Make the tree insert values for you

A comparator’s sign selects the branch. Recursive mutation helpers return the subtree root so callers can reconnect links. The plain rebalance hook returns the same node; AVL will override it. Cached height is unused in the plain tree.

The complete class is defined once below, including methods demonstrated in sections 4–6. Numeric examples require finite numbers: NaN and Infinity subtraction do not define a consistent ordering.

In [ ]:
const compareNumbers = (a, b)=>a - b;
class SearchNode {
    data;
    left = null;
    right = null;
    height = 0;
    constructor(data){
        this.data = data;
    }
}
class BinarySearchTree {
    compare;
    root = null;
    constructor(compare){
        this.compare = compare;
    }
    insert(value) {
        this.root = this.insertNode(this.root, value);
    }
    insertNode(node, value) {
        if (node === null) return new SearchNode(value);
        const order = this.compare(value, node.data);
        if (order < 0) {
            node.left = this.insertNode(node.left, value);
        } else if (order > 0) {
            node.right = this.insertNode(node.right, value);
        } else {
            return node;
        }
        return this.rebalance(node);
    }
    rebalance(node) {
        return node;
    }
    has(value) {
        let node = this.root;
        while(node !== null){
            const order = this.compare(value, node.data);
            if (order === 0) return true;
            node = order < 0 ? node.left : node.right;
        }
        return false;
    }
    minNode(node) {
        while(node.left !== null)node = node.left;
        return node;
    }
    min() {
        if (this.root === null) return undefined;
        return this.minNode(this.root).data;
    }
    max() {
        let node = this.root;
        if (node === null) return undefined;
        while(node.right !== null)node = node.right;
        return node.data;
    }
    traverse(visit, order = 'in') {
        if (!['in', 'pre', 'post'].includes(order)) throw new RangeError('Unknown traversal order');
        const walk = (node)=>{
            if (node === null) return;
            if (order === 'pre') visit(node.data);
            walk(node.left);
            if (order === 'in') visit(node.data);
            walk(node.right);
            if (order === 'post') visit(node.data);
        };
        walk(this.root);
    }
    values() {
        const result = [];
        this.traverse((value)=>{
            result.push(value);
        });
        return result;
    }
    remove(value) {
        this.root = this.removeNode(this.root, value);
    }
    removeNode(node, value) {
        if (node === null) return null;
        const order = this.compare(value, node.data);
        if (order < 0) {
            node.left = this.removeNode(node.left, value);
        } else if (order > 0) {
            node.right = this.removeNode(node.right, value);
        } else {
            if (node.left === null) return node.right;
            if (node.right === null) return node.left;
            const successor = this.minNode(node.right);
            node.data = successor.data;
            node.right = this.removeNode(node.right, successor.data);
        }
        return this.rebalance(node);
    }
}

In [ ]:
const tree = new BinarySearchTree(compareNumbers);
originalValues.forEach(value => tree.insert(value));
console.log("Inserted seven values:", tree.values());

### Checkpoint 3

Trace the insertion of 20. Where does it attach?

Predict the result before running the worked solution.

**Worked solution.** 20 goes right from 11, right from 15, and attaches in the empty right branch.

In [ ]:
(() => {
const assert = require("node:assert/strict");
assert.equal(tree.root.data, 11);
assert.equal(tree.root.right.data, 15);
assert.equal(tree.root.right.right.data, 20);
console.log('Insertion path: 11 → 15 → 20');
})();

## 4. Check what insertion built

Lookup for 13 and 12 both follows 11 → 15 → 13. 12 then reaches an empty left branch. Minimum follows left links, maximum right links; neither requires a leaf. Empty extrema return undefined, which is ambiguous if undefined itself is a stored value.

In [ ]:
console.log(tree.has(13), tree.has(12)); // true false
console.log(tree.min(), tree.max()); // 5 20

### Checkpoint 4

Insert 12 temporarily and confirm that lookup changes. Restore the original seven values.

Predict the result before running the worked solution.

**Worked solution.** A separate tree keeps the experiment isolated. The extra key changes has(12) to true; removing it restores false.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const temporary = new BinarySearchTree(compareNumbers);
originalValues.forEach(v => temporary.insert(v));
assert.equal(temporary.has(12), false);
temporary.insert(12);
assert.equal(temporary.has(12), true);
temporary.remove(12);
assert.equal(temporary.has(12), false);
assert.deepEqual(temporary.values(), tree.values());
console.log('has(12): false → true → false');
})();

## 5. Visit every value

In-order visits left, node, right; pre-order visits node before children; post-order visits children before node. Each visits O(n) nodes with O(h + 1) recursive stack space, plus any collected output. In-order is sorted under the comparator. A traversal alone needs additional structural information to reconstruct an arbitrary tree.

In [ ]:
console.log(tree.values());
const pre = [], post = [];
tree.traverse(value => pre.push(value), 'pre');
tree.traverse(value => post.push(value), 'post');
console.log(pre);
console.log(post);

### Checkpoint 5

Where does 11 appear in each traversal?

Predict the result before running the worked solution.

**Worked solution.** Fourth in-order, first in pre-order, last in post-order.

In [ ]:
(() => {
const assert = require("node:assert/strict");
assert.deepEqual(tree.values(), [5, 7, 9, 11, 13, 15, 20]);
assert.deepEqual(pre, [11, 7, 5, 9, 15, 13, 20]);
assert.deepEqual(post, [5, 9, 7, 13, 20, 15, 11]);
console.log('Positions of 11:', tree.values().indexOf(11) + 1, pre.indexOf(11) + 1, post.indexOf(11) + 1);
})();

## 6. Remove a value and close the gap

Removing a leaf returns null; removing a one-child node returns the surviving child. For two children, copy the in-order successor (minimum in the right subtree), then remove its original occurrence. Each surviving ancestor passes through rebalance on the return path.

In [ ]:
tree.remove(11);
console.log(tree.values()); // [5, 7, 9, 13, 15, 20]

### Checkpoint 6

Check the root deletion. Separately remove 5, then 7 from the original tree.

Predict the result before running the worked solution.

**Worked solution.** 13 appears exactly once after removing 11. In the separate experiment, removing leaf 5 leaves 7 with one child, so removing 7 promotes 9.

In [ ]:
(() => {
const assert = require("node:assert/strict");
assert.deepEqual(tree.values(), [5, 7, 9, 13, 15, 20]);
const experiment = new BinarySearchTree(compareNumbers);
originalValues.forEach(v => experiment.insert(v));
experiment.remove(5);
experiment.remove(7);
assert.equal(experiment.root.left.data, 9);
assert.deepEqual(experiment.values(), [9, 11, 13, 15, 20]);
console.log('After removing 5 and 7:', experiment.values());
})();

## 7. See the tree get tall, then repair it

Sorted insertion makes a plain BST a chain. AVL keeps each balance factor (left height minus right height) between −1 and 1. A leaf has height 0; an empty subtree has height −1.

| Heavy path | Repair |
| --- | --- |
| Left-left | Right rotation |
| Right-right | Left rotation |
| Left-right | Left rotation on child, then right |
| Right-left | Right rotation on child, then left |

Rotations preserve in-order keys and return the new subtree root. Update the demoted node’s height before the pivot’s. After deletion, a heavy child with balance factor 0 needs a single rotation; repair can be needed at multiple ancestors.

In [ ]:
class AVLTree extends BinarySearchTree {
    height(node) {
        return node === null ? -1 : node.height;
    }
    update(node) {
        node.height = 1 + Math.max(this.height(node.left), this.height(node.right));
    }
    balanceFactor(node) {
        return this.height(node.left) - this.height(node.right);
    }
    rotateRight(node) {
        const pivot = node.left;
        if (pivot === null) throw new Error('Missing left pivot');
        node.left = pivot.right;
        pivot.right = node;
        this.update(node);
        this.update(pivot);
        return pivot;
    }
    rotateLeft(node) {
        const pivot = node.right;
        if (pivot === null) throw new Error('Missing right pivot');
        node.right = pivot.left;
        pivot.left = node;
        this.update(node);
        this.update(pivot);
        return pivot;
    }
    rebalance(node) {
        this.update(node);
        const balance = this.balanceFactor(node);
        if (balance > 1 && node.left !== null) {
            if (this.balanceFactor(node.left) < 0) {
                node.left = this.rotateLeft(node.left);
            }
            return this.rotateRight(node);
        }
        if (balance < -1 && node.right !== null) {
            if (this.balanceFactor(node.right) > 0) {
                node.right = this.rotateRight(node.right);
            }
            return this.rotateLeft(node);
        }
        return node;
    }
}

In [ ]:
const avl = new AVLTree(compareNumbers);
[
    5,
    7,
    9,
    11,
    13,
    15,
    20
].forEach((value)=>{
    avl.insert(value);
});
console.log(avl.values());
const avl2 = new AVLTree(compareNumbers);
[
    11,
    7,
    15,
    5,
    9,
    13,
    20
].forEach((value)=>{
    avl2.insert(value);
});
avl2.remove(11);
console.log(avl2.values());

### Checkpoint 7

Insert 15, 11, 7. What is the repaired pre-order? Check deletion of 11 in the full AVL tree.

Predict the result before running the worked solution.

**Worked solution.** The right rotation gives [11, 7, 15]. Deleting 11 from the full AVL tree gives [5, 7, 9, 13, 15, 20].

In [ ]:
(() => {
const assert = require("node:assert/strict");
const three = new AVLTree(compareNumbers);
[15, 11, 7].forEach(v => three.insert(v));
const order = [];
three.traverse(v => order.push(v), 'pre');
assert.deepEqual(order, [11, 7, 15]);
assert.deepEqual(avl2.values(), [5, 7, 9, 13, 15, 20]);
assert.equal(three.root.height, 1);
console.log('Rotation pre-order:', order, 'After deletion:', avl2.values());
})();

## 8. Balance by colour instead of height

In this left-leaning red-black tree, the root and absent children count as black, red links lean left, red nodes cannot have red children, and all paths to absent descendants have equal black counts. Rotations transfer colours; flips move a conflict upward. Inserting 11, 15, 7 exercises a left rotation, a right rotation, and a colour flip.

This class provides insertion, lookup, and sorted traversal. Red-black removal requires a separate colour-aware algorithm; plain BST deletion would not preserve these invariants.

In [ ]:
class RedBlackNode {
    data;
    left = null;
    right = null;
    color = 'red';
    constructor(data){
        this.data = data;
    }
}
class RedBlackTree {
    compare;
    root = null;
    constructor(compare){
        this.compare = compare;
    }
    isRed(node) {
        return node !== null && node.color === 'red';
    }
    rotateLeft(node) {
        const pivot = node.right;
        if (pivot === null) throw new Error('Missing right pivot');
        node.right = pivot.left;
        pivot.left = node;
        pivot.color = node.color;
        node.color = 'red';
        return pivot;
    }
    rotateRight(node) {
        const pivot = node.left;
        if (pivot === null) throw new Error('Missing left pivot');
        node.left = pivot.right;
        pivot.right = node;
        pivot.color = node.color;
        node.color = 'red';
        return pivot;
    }
    flipColors(node) {
        const left = node.left;
        const right = node.right;
        if (left === null || right === null) {
            throw new Error('Color flip requires two children');
        }
        node.color = node.color === 'red' ? 'black' : 'red';
        left.color = left.color === 'red' ? 'black' : 'red';
        right.color = right.color === 'red' ? 'black' : 'red';
    }
    insert(value) {
        this.root = this.insertNode(this.root, value);
        if (this.root !== null) this.root.color = 'black';
    }
    insertNode(node, value) {
        if (node === null) return new RedBlackNode(value);
        const order = this.compare(value, node.data);
        if (order < 0) {
            node.left = this.insertNode(node.left, value);
        } else if (order > 0) {
            node.right = this.insertNode(node.right, value);
        } else {
            return node;
        }
        if (this.isRed(node.right) && !this.isRed(node.left)) {
            node = this.rotateLeft(node);
        }
        const left = node.left;
        if (left !== null && this.isRed(left) && this.isRed(left.left)) {
            node = this.rotateRight(node);
        }
        if (this.isRed(node.left) && this.isRed(node.right)) {
            this.flipColors(node);
        }
        return node;
    }
    has(value) {
        let node = this.root;
        while(node !== null){
            const order = this.compare(value, node.data);
            if (order === 0) return true;
            node = order < 0 ? node.left : node.right;
        }
        return false;
    }
    values() {
        const result = [];
        const walk = (node)=>{
            if (node === null) return;
            walk(node.left);
            result.push(node.data);
            walk(node.right);
        };
        walk(this.root);
        return result;
    }
}
const rb = new RedBlackTree(compareNumbers);
[
    11,
    7,
    15,
    5,
    9,
    13,
    20
].forEach((value)=>{
    rb.insert(value);
});
console.log(rb.values(), rb.has(13));

### Checkpoint 8

Insert 13 again. Does the sorted result contain a duplicate?

Predict the result before running the worked solution.

**Worked solution.** No. Comparator equality keeps the existing value.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const before = rb.values();
rb.insert(13);
assert.deepEqual(rb.values(), before);
assert.equal(rb.values().filter(v => v === 13).length, 1);
assert.equal(rb.root.color, 'black');
console.log(rb.values(), rb.has(13));
})();

## 9. Connect the example to the running time

| Operation | Plain BST worst case | AVL worst case | Red-black worst case |
| --- | --- | --- | --- |
| Lookup/insertion | O(n) | O(log n) | O(log n) |
| Removal | O(n) | O(log n) | O(log n) with colour-aware deletion |
| Traversal | O(n) | O(n) | O(n) |
| Node storage | O(n) | O(n) | O(n) |

Assume constant-time comparisons and visits. Path work takes O(h + 1). Recursive mutation and traversal use O(h + 1) stack space; iterative lookup and extrema use O(1) extra space. A plain BST may have logarithmic expected height under random insertion order; balanced trees guarantee logarithmic height. Map/Set may suit equality lookup; disk indexes often use B-trees or B+ trees.

### Checkpoint 9

How many nodes are visited when searching for the last of 100 ascending inserts?

Predict the result before running the worked solution.

**Worked solution.** 100 nodes: sorted insertion creates a chain of height 99.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const chain = new BinarySearchTree(compareNumbers);
for (let v = 1; v <= 100; v++) chain.insert(v);
let node = chain.root, visits = 0;
while (node !== null) { visits++; if (node.data === 100) break; node = node.right; }
assert.equal(visits, 100);
assert.equal(nodeHeight(chain.root), 99);
console.log({ visits, height: nodeHeight(chain.root) });
})();

## 10. Reuse the ideas in new examples

### Order records by name

Compare surname, then first name, then ID using English collation. Equality concerns all three fields; enforce ID-only uniqueness separately. JavaScript does not enforce the source guide’s readonly fields: preserve ordering keys while stored.

In [ ]:
const byName = (a, b)=>{
    return a.lastName.localeCompare(b.lastName, 'en') || a.firstName.localeCompare(b.firstName, 'en') || a.id - b.id;
};
const students = new BinarySearchTree(byName);
[
    {
        id: 11,
        firstName: 'Darcy',
        lastName: 'Vega'
    },
    {
        id: 7,
        firstName: 'Tory',
        lastName: 'Vega'
    },
    {
        id: 5,
        firstName: 'Caleb',
        lastName: 'Altair'
    }
].forEach((student)=>{
    students.insert(student);
});
const roster = [];
students.traverse((student)=>{
    roster.push(`${student.id}: ${student.lastName}, ${student.firstName}`);
});
console.log(roster);

### Checkpoint 10

Where would Darcy Vega with ID 3 go?

Predict the result before running the worked solution.

**Worked solution.** Before Darcy Vega with ID 11, after Caleb Altair. Only the ID tie-break differs.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const sample = new BinarySearchTree(byName);
students.values().forEach(v => sample.insert(v));
sample.insert({ id: 3, firstName: 'Darcy', lastName: 'Vega' });
assert.deepEqual(sample.values().map(v => v.id), [5, 3, 11, 7]);
assert.deepEqual(roster, ['5: Altair, Caleb', '11: Vega, Darcy', '7: Vega, Tory']);
console.log(sample.values());
})();

### Walk a directory tree

Here a branch means containment, not key order. Sum file sizes recursively; visit contents before the directory to form a post-order deletion plan. This only returns proposed paths in memory. Assume an acyclic hierarchy with no shared entries. Full paths and arrays are copied while returning.

In [ ]:
function totalBytes(entry) {
    if (entry.kind === 'file') return entry.bytes;
    return entry.children.reduce((sum, child)=>sum + totalBytes(child), 0);
}
function deletionPlan(entry, parent = '') {
    const path = `${parent}/${entry.name}`;
    if (entry.kind === 'file') return [
        path
    ];
    const result = [];
    for (const child of entry.children){
        result.push(...deletionPlan(child, path));
    }
    result.push(path);
    return result;
}
const project = {
    kind: 'directory',
    name: 'project',
    children: [
        {
            kind: 'file',
            name: 'README.md',
            bytes: 200
        },
        {
            kind: 'directory',
            name: 'src',
            children: [
                {
                    kind: 'file',
                    name: 'app.ts',
                    bytes: 800
                }
            ]
        }
    ]
};
console.log(totalBytes(project));
console.log(deletionPlan(project));

### Checkpoint 11

What are the final two paths in the deletion plan?

Predict the result before running the worked solution.

**Worked solution.** /project/src and /project. app.ts must appear before its containing directory.

In [ ]:
(() => {
const assert = require("node:assert/strict");
const plan = deletionPlan(project);
assert.equal(totalBytes(project), 1000);
assert.deepEqual(plan.slice(-2), ['/project/src', '/project']);
assert.ok(plan.indexOf('/project/src/app.ts') < plan.indexOf('/project/src'));
console.log(plan);
})();

### Learn a single decision

A decision stump learns one threshold from labelled samples. Training tries distinct neighboring waiting times and chooses a split only when it reduces mistakes. Tied majorities predict routine; tied scores keep the earlier choice. Prediction follows the learned branch.

The synthetic data yields threshold 7, routine on the left, urgent on the right. This trainer scans and copies partitions, taking O(n²) time after sorting and O(n) peak temporary storage. Prediction takes O(d + 1) time and constant auxiliary space for depth d. Separate labelled evaluation data would be needed to assess accuracy.

In [ ]:
function predict(tree, waitingHours) {
    if (!Number.isFinite(waitingHours) || waitingHours < 0) {
        throw new Error('Waiting hours must be finite and nonnegative');
    }
    let node = tree;
    while(node.kind === 'split'){
        node = waitingHours <= node.threshold ? node.left : node.right;
    }
    return node.prediction;
}
function majority(samples) {
    const urgent = samples.filter((sample)=>sample.priority === 'urgent').length;
    return urgent > samples.length / 2 ? 'urgent' : 'routine';
}
function mistakes(samples, label) {
    return samples.filter((sample)=>sample.priority !== label).length;
}
function trainStump(samples) {
    if (samples.length === 0) throw new Error('Training data is empty');
    if (samples.some((sample)=>!Number.isFinite(sample.waitingHours) || sample.waitingHours < 0)) {
        throw new Error('Invalid training hours');
    }
    if (samples.some(sample => !['routine', 'urgent'].includes(sample.priority))) {
        throw new Error('Invalid training priority');
    }
    const sorted = [
        ...samples
    ].sort((a, b)=>a.waitingHours - b.waitingHours);
    const baseline = majority(sorted);
    let best = {
        kind: 'leaf',
        prediction: baseline
    };
    let bestErrors = mistakes(sorted, baseline);
    for(let i = 1; i < sorted.length; i++){
        const before = sorted[i - 1];
        const after = sorted[i];
        if (before === undefined || after === undefined) continue;
        if (before.waitingHours === after.waitingHours) continue;
        const left = sorted.slice(0, i);
        const right = sorted.slice(i);
        const leftLabel = majority(left);
        const rightLabel = majority(right);
        const errors = mistakes(left, leftLabel) + mistakes(right, rightLabel);
        if (errors < bestErrors) {
            bestErrors = errors;
            best = {
                kind: 'split',
                threshold: before.waitingHours / 2 + after.waitingHours / 2,
                left: {
                    kind: 'leaf',
                    prediction: leftLabel
                },
                right: {
                    kind: 'leaf',
                    prediction: rightLabel
                }
            };
        }
    }
    return best;
}
const training = [
    {
        waitingHours: 0,
        priority: 'routine'
    },
    {
        waitingHours: 2,
        priority: 'routine'
    },
    {
        waitingHours: 4,
        priority: 'routine'
    },
    {
        waitingHours: 6,
        priority: 'routine'
    },
    {
        waitingHours: 8,
        priority: 'urgent'
    },
    {
        waitingHours: 10,
        priority: 'urgent'
    },
    {
        waitingHours: 12,
        priority: 'urgent'
    },
    {
        waitingHours: 14,
        priority: 'urgent'
    }
];
const model = trainStump(training);
console.log(model);
console.log(predict(model, 3));
console.log(predict(model, 9));

### Checkpoint 12

What is the threshold? Add an urgent sample at 5 hours: why can one threshold no longer classify every row?

Predict the result before running the worked solution.

**Worked solution.** The original threshold is 7 with zero training mistakes (versus four for a routine-only baseline). An urgent sample at 5 lies before the routine sample at 6, followed by urgent samples at 8 and above: one split cannot separate all three regions.

In [ ]:
(() => {
const assert = require("node:assert/strict");
assert.equal(model.threshold, 7);
assert.equal(training.filter(s => predict(model, s.waitingHours) !== s.priority).length, 0);
assert.equal(mistakes(training, 'routine'), 4);
const extended = [...training, { waitingHours: 5, priority: 'urgent' }];
const changed = trainStump(extended);
const errors = extended.filter(s => predict(changed, s.waitingHours) !== s.priority).length;
assert.equal(errors, 1);
console.log({ originalThreshold: model.threshold, changedModel: changed, trainingErrors: errors });
})();

## 11. Practise one idea at a time

Try the three exercises before running the implementations and worked checks below.

1. Replace recursive in-order traversal with an explicit stack. Visit the tree rooted at 2 with children 1 and 3.
2. Validate ancestor bounds, not only immediate children. Reject 10 with right child 15 whose left child is 6.
3. Rebuild [5, 7, 9, 11, 13, 15, 20] by choosing the middle of each half-open range. The input must be dense, strictly increasing, and finite.

The exercise node uses val; the search-tree node uses data. In-order is O(n) time, O(h + 1) stack plus output. Validation is O(n) time and O(h + 1) stack. Construction is O(n) time and output space with O(log n) stack.

In [ ]:
class TreeNode {
    val;
    left;
    right;
    constructor(val = 0, left = null, right = null){
        this.val = val;
        this.left = left;
        this.right = right;
    }
}
function inorderTraversal(root) {
    const result = [];
    const stack = [];
    let current = root;
    while(current !== null || stack.length > 0){
        while(current !== null){
            stack.push(current);
            current = current.left;
        }
        const next = stack.pop();
        if (next === undefined) break;
        result.push(next.val);
        current = next.right;
    }
    return result;
}
function isValidBST(root) {
    const valid = (node, lower, upper)=>{
        if (node === null) return true;
        if (!Number.isFinite(node.val)) return false;
        if (lower !== undefined && node.val <= lower) return false;
        if (upper !== undefined && node.val >= upper) return false;
        return valid(node.left, lower, node.val) && valid(node.right, node.val, upper);
    };
    return valid(root, undefined, undefined);
}
function sortedArrayToBST(values) {
    const build = (start, end)=>{
        if (start >= end) return null;
        const middle = start + Math.floor((end - start) / 2);
        const value = values[middle];
        if (value === undefined) throw new Error('Invalid array bounds');
        return new TreeNode(value, build(start, middle), build(middle + 1, end));
    };
    return build(0, values.length);
}
const exerciseRoot = new TreeNode(2, new TreeNode(1), new TreeNode(3));
console.log(inorderTraversal(exerciseRoot));
console.log(isValidBST(exerciseRoot));
console.log(isValidBST(new TreeNode(10, null, new TreeNode(15, new TreeNode(6), null))));
console.log(inorderTraversal(sortedArrayToBST([
    1,
    2,
    3,
    4,
    5,
    6,
    7
])));

### Worked exercise checks

The stack visits [1, 2, 3]; the bound violation fails; middle-based construction restores root 11, children 7 and 15. Empty inputs work in all three cases.

In [ ]:
(() => {
 const assert = require('node:assert/strict');
 assert.deepEqual(inorderTraversal(exerciseRoot), [1, 2, 3]);
 assert.deepEqual(inorderTraversal(null), []);
 assert.equal(isValidBST(null), true);
 assert.equal(isValidBST(new TreeNode(10, null, new TreeNode(15, new TreeNode(6)))), false);
 assert.equal(isValidBST(new TreeNode(2, new TreeNode(2))), false);
 assert.equal(isValidBST(new TreeNode(Infinity)), false);
 const rebuilt = sortedArrayToBST([5, 7, 9, 11, 13, 15, 20]);
 assert.equal(rebuilt.val, 11);
 assert.equal(rebuilt.left.val, 7);
 assert.equal(rebuilt.right.val, 15);
 assert.deepEqual(inorderTraversal(rebuilt), [5, 7, 9, 11, 13, 15, 20]);
 assert.equal(isValidBST(rebuilt), true);
 assert.equal(sortedArrayToBST([]), null);
 console.log('All three exercise solutions passed.');
})();

## 12. Check the structure as well as the output

Sorted output alone cannot prove balance. Compare mixed updates with a reference Set, compute actual AVL heights and compare caches, and verify all left-leaning red-black colour rules. Include sorted, descending, and deterministic shuffled insertions.

The JavaScript adaptation rejects unsupported traversal orders and invalid decision labels at runtime. Compile-time rejection of wrong generic types or missing record fields requires the TypeScript setup in the supplied guide.

In [ ]:
(() => {
 const assert = require('node:assert/strict');
 function ordered(node, lower = -Infinity, upper = Infinity) {
  if (node === null) return;
  assert.ok(Number.isFinite(node.data) && node.data > lower && node.data < upper);
  ordered(node.left, lower, node.data); ordered(node.right, node.data, upper);
 }
 function avlHeight(node) {
  if (node === null) return -1;
  const left = avlHeight(node.left), right = avlHeight(node.right);
  assert.ok(Math.abs(left - right) <= 1, 'AVL balance');
  const height = 1 + Math.max(left, right);
  assert.equal(node.height, height, 'Cached AVL height');
  return height;
 }
 function blackHeight(node) {
  if (node === null) return 1;
  assert.ok(['red', 'black'].includes(node.color));
  assert.ok(node.right === null || node.right.color !== 'red', 'Red links lean left');
  if (node.color === 'red') {
   assert.ok(node.left === null || node.left.color === 'black');
   assert.ok(node.right === null || node.right.color === 'black');
  }
  const left = blackHeight(node.left), right = blackHeight(node.right);
  assert.equal(left, right, 'Equal black heights');
  return left + (node.color === 'black' ? 1 : 0);
 }
 let seed = 10;
 const random = () => ((seed = (Math.imul(seed, 1664525) + 1013904223) >>> 0) / 2 ** 32);
 const ascending = Array.from({ length: 80 }, (_, i) => i);
 const shuffled = ascending.slice();
 for (let i = shuffled.length - 1; i > 0; i--) {
  const j = Math.floor(random() * (i + 1)); [shuffled[i], shuffled[j]] = [shuffled[j], shuffled[i]];
 }
 for (const sequence of [ascending, ascending.slice().reverse(), shuffled]) {
  for (const Tree of [BinarySearchTree, AVLTree]) {
   const sample = new Tree(compareNumbers), reference = new Set();
   const verify = () => {
    assert.deepEqual(sample.values(), [...reference].sort(compareNumbers));
    ordered(sample.root);
    if (Tree === AVLTree) avlHeight(sample.root);
   };
   for (const value of sequence) { sample.insert(value); reference.add(value); verify(); }
   sample.insert(13); verify();
   sample.remove(999); verify();
   assert.equal(sample.has(999), false);
   for (let i = 0; i < 200; i++) {
    const value = Math.floor(random() * 100);
    if (random() < 0.5) { sample.insert(value); reference.add(value); }
    else { sample.remove(value); reference.delete(value); }
    verify();
   }
   for (const value of [...reference]) { sample.remove(value); reference.delete(value); verify(); }
   assert.equal(sample.min(), undefined); assert.equal(sample.max(), undefined);
   assert.equal(sample.root, null);
  }
  const sample = new RedBlackTree(compareNumbers), reference = new Set();
  for (const value of [...sequence, ...sequence]) {
   sample.insert(value); reference.add(value);
   assert.deepEqual(sample.values(), [...reference].sort(compareNumbers));
   assert.equal(sample.root.color, 'black'); ordered(sample.root); blackHeight(sample.root);
  }
 }
 // Explicit LL, RR, LR, RL rotation cases.
 for (const values of [[15,11,7], [7,11,15], [15,7,11], [7,15,11]]) {
  const sample = new AVLTree(compareNumbers);
  values.forEach(v => sample.insert(v));
  assert.equal(sample.root.data, 11); avlHeight(sample.root);
 }
 // Deletion with a heavy child whose balance factor is zero.
 const deletion = new AVLTree(compareNumbers);
 [4,2,5,1,3].forEach(v => deletion.insert(v)); deletion.remove(5);
 assert.equal(deletion.root.data, 2); avlHeight(deletion.root);
 assert.throws(() => tree.traverse(() => {}, 'sideways'), /traversal order/);
 assert.deepEqual(tree.values(), [5,7,9,13,15,20]);
 assert.equal(totalBytes(project), 1000);
 assert.deepEqual(deletionPlan(project), ['/project/README.md','/project/src/app.ts','/project/src','/project']);
 assert.equal(predict(model, 3), 'routine'); assert.equal(predict(model, 9), 'urgent');
 assert.throws(() => trainStump([]), /empty/);
 assert.throws(() => trainStump([{waitingHours: -1, priority: 'routine'}]), /hours/);
 assert.throws(() => trainStump([{waitingHours: 1, priority: 'unknown'}]), /priority/);
 assert.throws(() => predict(model, NaN), /finite/);
 assert.equal(trainStump([{waitingHours: 2, priority: 'routine'}, {waitingHours: 2, priority: 'urgent'}]).prediction, 'routine');
 console.log('All Chapter 10 checks passed.');
})();

## Review

Comparators make one search rule reusable. Returning subtree roots reconnects insertion, deletion, and rotations. Heights and colour invariants reveal structure that sorted output cannot. Folder containment and decision branches demonstrate why choosing the right tree model matters.

Adapted from the supplied Chapter 10 Trees guide. The source TypeScript contracts are described above; executable cells use JavaScript. The next chapter uses heap ordering to retrieve priority items.